# Python Mastery
### Essential concepts, elaborated theory, and worked examples

A professional reference notebook for the Python language itself: the execution model, data model, control flow, functions, collections, errors, files, objects, and the idioms that separate correct code from durable code.

Each section states the idea, then shows it. Outputs below were captured from a live interpreter, so the printed results match the source.

**How to use this notebook**
- Read the theory block before running the cell beneath it.
- Cells are ordered. Later sections assume earlier ideas.
- Re-run any cell after editing it. Shared state lives in this kernel.
- Prefer the *why* over memorising syntax. Syntax is cheap; the data model is not.

**What this notebook covers**
1. The execution model and the data model
2. Names, binding, and mutability
3. Scalar types: `int`, `float`, `bool`, `None`, `complex`
4. Operators, precedence, and truthiness
5. Control flow
6. Functions, scope, closures, and argument forms
7. Strings and text
8. Lists, tuples, sets, dictionaries, and the collections protocol
9. Comprehensions, iterators, and generators
10. Exceptions
11. Files and context managers
12. Object-oriented programming
13. Decorators and functional tools
14. Modules and a curated standard library
15. Type hints and modern idioms
16. A short capstone


## 1. The execution model

Python is an interpreted, dynamically typed, strongly typed language with a reference implementation called CPython. Source is compiled to bytecode (`.pyc`), then executed by a stack-based virtual machine. You rarely see that step; `python file.py` does both.

Three facts govern almost every later topic.

**Everything is an object.** Integers, functions, modules, classes, and even `None` are objects with an identity, a type, and a value. `id(x)` is the identity. `type(x)` is the type. The value is what the object means.

**Names are labels, not boxes.** Assignment does not copy a value into a variable. It binds a name to an object in a namespace. `a = b` makes `a` and `b` refer to the same object until one of them is rebound.

**Typing is dynamic and strong.** A name may be rebound to an object of another type at any moment (dynamic). Operations still refuse nonsensical combinations such as `"3" + 4` (strong). There is no silent coercion between `str` and `int`.

CPython also uses reference counting plus a cyclic garbage collector. You do not free memory manually. You do need to know when two names share an object, because mutating that object is visible through every name bound to it.


In [1]:
x = 10
print("type:", type(x).__name__)
print("identity:", id(x))
print("value:", x)

def add(a, b):
    return a + b

print("code object names:", add.__code__.co_varnames)
print("constants:", add.__code__.co_consts)


type: int
identity: 139868709847568
value: 10
code object names: ('a', 'b')
constants: (None,)


## 2. Names, binding, and mutability

A namespace is a mapping from names to objects. Python has several: the local namespace of a function, the enclosing namespaces of nested functions, the module global namespace, and the built-in namespace. Name lookup follows the LEGB rule: Local, Enclosing, Global, Built-in.

Assignment is local unless declared otherwise. `global` rebinds a module-level name. `nonlocal` rebinds a name in an enclosing function. Reading a name never needs those keywords; writing does, if the name is not local.

**Mutable vs immutable is a property of the object, not of the name.**
- Immutable: `int`, `float`, `complex`, `bool`, `str`, `tuple`, `frozenset`, `bytes`, `None`.
- Mutable: `list`, `dict`, `set`, `bytearray`, and most user-defined objects.

Rebinding (`a = a + [1]`) and mutating (`a.append(1)`) are different operations. The first creates or looks up a new object and points the name at it. The second changes the object in place, so every alias sees the change.

`is` compares identity. `==` compares value, by calling `__eq__`. Use `is` for `None`, `True`, and `False`. Use `==` for values.


In [2]:
a = [1, 2, 3]
b = a            # alias, not a copy
c = a.copy()     # shallow copy
a.append(4)
print("a:", a)
print("b:", b, "(alias of a)")
print("c:", c, "(shallow copy)")
print("a is b:", a is b)
print("a == c:", a == c)

n = 256
m = 256
print("small ints may be interned:", n is m)
s1 = "hello"
s2 = "hello"
print("interned string identity:", s1 is s2)
print("None check idiom:", s1 is not None)


a: [1, 2, 3, 4]
b: [1, 2, 3, 4] (alias of a)
c: [1, 2, 3] (shallow copy)
a is b: True
a == c: False
small ints may be interned: True
interned string identity: True
None check idiom: True


## 3. Scalar types

### Integers
`int` is arbitrary precision. There is no fixed overflow at 2^63. Literals may use `_` as a visual separator, and prefixes `0b`, `0o`, `0x`.

### Floats
`float` is IEEE-754 binary64. It cannot represent most decimal fractions exactly. Compare with a tolerance, or use `decimal.Decimal` for money and exact base-10 work.

### Booleans
`bool` is a subclass of `int`. `True == 1` and `False == 0` are true. That is occasionally useful and often a trap. Prefer explicit booleans in APIs.

### None
`None` is the sole instance of `NoneType`. It means "no value", not zero and not false-as-data. Always test it with `is`.

### Complex
`complex` stores a real and an imaginary part. `j` is the literal suffix. Useful in signal processing; rare in everyday application code.


In [3]:
print(10_000_000)
print(0b1010, 0o17, 0xFF)
print(2 ** 100)                 # arbitrary precision

x = 0.1 + 0.2
print(x)
print(round(x, 10) == 0.3)

from decimal import Decimal
print(Decimal("0.1") + Decimal("0.2"))

print(isinstance(True, int), True + True)
print(type(None).__name__)

z = 3 + 4j
print(z.real, z.imag, abs(z))


10000000
10 15 255
1267650600228229401496703205376
0.30000000000000004
True
0.3
True 2
NoneType
3.0 4.0 5.0


## 4. Operators, precedence, and truthiness

Arithmetic operators follow normal mathematical precedence: `**` binds tightest, then unary `+`/`-`, then `*`, `/`, `//`, `%`, then `+` and `-`. Comparisons chain: `1 < x < 10` is `1 < x and x < 10`, with `x` evaluated once.

Division is the operator people misread.
- `/` is true division and always returns a `float`.
- `//` is floor division.
- `%` is the remainder, with the sign of the divisor.

Bitwise operators (`&`, `|`, `^`, `~`, `<<`, `>>`) work on the integer's two's-complement representation. They are not boolean operators. Boolean logic uses `and`, `or`, and `not`, which short-circuit and return the last evaluated operand, not necessarily `True` or `False`.

**Truthiness.** An object is false when its `__bool__` returns false, or, failing that, when `__len__` returns 0. False values you should know: `None`, `False`, numeric zero, empty containers, and empty strings. Everything else is true.


In [4]:
print("true division:", 7 / 2)
print("floor division:", 7 // 2, (-7) // 2)
print("remainder:", 7 % 2, (-7) % 2)
print("power:", 2 ** 10)
print("chained compare:", 1 < 3 < 5)

print("and returns operand:", 0 and "skipped", "ok" and "also ok")
print("or returns operand:", "" or "fallback")

values = [None, False, 0, 0.0, "", [], {}, set(), "x", [0], 1]
for v in values:
    print(repr(v), "->", bool(v))

flags = 0b1010
print("bit set:", bool(flags & 0b1000), "shifted:", bin(flags << 1))


true division: 3.5
floor division: 3 -4
remainder: 1 1
power: 1024
chained compare: True
and returns operand: 0 also ok
or returns operand: fallback
None -> False
False -> False
0 -> False
0.0 -> False
'' -> False
[] -> False
{} -> False
set() -> False
'x' -> True
[0] -> True
1 -> True
bit set: True shifted: 0b10100


## 5. Control flow

`if` / `elif` / `else` select a branch. There is no `switch` statement before 3.10; from 3.10 onward, `match` / `case` performs structural pattern matching. It matches patterns, not just constants: literals, sequences, mappings, classes, and capture names with guards.

`while` repeats while a condition holds. `for` iterates over an iterable. It does not count unless you hand it a range. `range` is lazy: `range(10**9)` does not allocate a billion integers.

`break` leaves the nearest loop. `continue` skips to the next iteration. A loop may have an `else` clause, which runs only if the loop was not left by `break`. That clause is the right tool for a search that may fail.

The walrus operator `:=` assigns inside an expression. Use it when a value is needed both as a condition and as a body input, not as a general replacement for assignment.


In [5]:
score = 86
if score >= 90:
    grade = "A"
elif score >= 80:
    grade = "B"
else:
    grade = "C"
print(grade)

for i, ch in enumerate("py", start=1):
    print(i, ch)

found = None
for n in [2, 4, 6, 9, 12]:
    if n % 2 == 1:
        found = n
        break
else:
    print("no odd number")
print("found:", found)

command = ("move", 3, "north")
match command:
    case ("move", n, direction) if n > 0:
        print(f"moving {n} steps {direction}")
    case ("stop",):
        print("stopped")
    case _:
        print("unknown")

raw_samples = ["alpha", "", "beta", "  ", "gamma"]
cleaned = []
for raw in raw_samples:
    if text := raw.strip():
        cleaned.append(text.upper())
print("walrus filter:", cleaned)


B
1 p
2 y
found: 9
moving 3 steps north
walrus filter: ['ALPHA', 'BETA', 'GAMMA']


## 6. Functions, scope, and argument forms

A function is an object created by `def` or `lambda`. `def` binds the name at execution time. Defaults are evaluated once, at definition time, not at call time. A mutable default is therefore shared across calls. Use `None` as the sentinel and create the container inside the body.

Argument forms, in order:
- positional-only, before `/`
- positional or keyword
- keyword-only, after `*`
- `*args` collects extra positional arguments as a tuple
- `**kwargs` collects extra keyword arguments as a dict

Return values are one object. Multiple values are a tuple that the caller may unpack.

**Scope.** A function sees locals, then enclosing scopes, then globals, then builtins. A closure is a nested function that keeps references to enclosing variables. Those variables are looked up when the inner function runs, not when it is defined. Late binding in loops is the classic surprise: capture the current value with a default argument.

`lambda` creates a one-expression function. It is appropriate as a key function. It is a poor substitute for `def` once the body needs a name, a docstring, or more than one statement.


In [6]:
def scale(values, factor=2, *, strict=False):
    """Return values scaled by factor. factor is keyword-capable; strict is keyword-only."""
    if strict and factor == 0:
        raise ValueError("factor must be non-zero in strict mode")
    return [v * factor for v in values]

print(scale([1, 2, 3]))
print(scale([1, 2, 3], 10))
print(scale([1, 2], factor=5, strict=True))

def collect(first, *rest, label="items"):
    return label, (first, *rest)

print(collect(1, 2, 3, label="nums"))

def adder(n):
    def add(x):
        return x + n
    return add

add10 = adder(10)
print(add10(7))

# Correct capture of loop variables
funcs = [lambda x, i=i: x + i for i in range(3)]
print([f(10) for f in funcs])

# Mutable default trap, and the fix
def bad(item, bucket=[]):
    bucket.append(item)
    return bucket

def good(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print("bad calls share state:", bad(1), bad(2))
print("good calls do not:", good(1), good(2))


[2, 4, 6]
[10, 20, 30]
[5, 10]
('nums', (1, 2, 3))
17
[10, 11, 12]
bad calls share state: [1, 2] [1, 2]
good calls do not: [1] [2]


## 7. Strings and text

A `str` is an immutable sequence of Unicode code points, not bytes. Indexing and slicing work as they do for any sequence. Slicing never raises for an out-of-range end; indexing does.

Encoding is the conversion from text to bytes. Decoding is the reverse. The boundary is the edge of your program: files, sockets, and subprocesses deal in bytes; your logic should deal in `str`. UTF-8 is the default you want unless a format demands otherwise.

Formatting, in order of preference for new code:
- f-strings for interpolation
- `str.format` when the template is supplied from outside
- `%` only when matching an old interface

Useful methods cluster around case, search, split/join, strip, and predicate tests (`isdigit`, `isalpha`). `join` is the correct way to concatenate many strings. Repeated `+` in a loop builds intermediate objects.


In [7]:
text = "  Python, 3.12  "
print(text.strip().lower())
print("python".title())
print("a,b,c".split(","))
print("-".join(["a", "b", "c"]))
print("py" in "python", "python".find("th"))
print("café".encode("utf-8"))
print(b"caf\xc3\xa9".decode("utf-8"))

name, version = "CPython", 3.12
print(f"{name} {version:.1f} | {version!r}")
print("{0} has {1} letters".format(name, len(name)))

path = "notes/chapter_01.md"
print(path.removesuffix(".md").replace("_", " "))
print("αβγ".casefold() == "ΑΒΓ".casefold())


python, 3.12
Python
['a', 'b', 'c']
a-b-c
True 2
b'caf\xc3\xa9'
café
CPython 3.1 | 3.12
CPython has 7 letters
notes/chapter 01
True


## 8. Lists, tuples, sets, and dictionaries

These four types cover most in-memory data.

| Type | Ordered | Mutable | Hashable | Typical use |
|---|---|---|---|---|
| `list` | yes | yes | no | sequences you will change |
| `tuple` | yes | no | yes, if items are | fixed records, dict keys |
| `set` | no | yes | no | membership, uniqueness |
| `dict` | insertion-ordered | yes | no | labelled records, indexes |

Hashability requires a stable hash and a compatible `__eq__`. Mutable containers are unhashable because their contents can change.

**Lists.** Amortised O(1) append. Insert or delete at the front is O(n). Slicing copies. `sort` is in place and stable; `sorted` returns a new list.

**Tuples.** A record, not a frozen list. Unpack them. A one-element tuple needs a trailing comma: `(1,)`.

**Sets.** Average O(1) membership. Operators `|`, `&`, `-`, `^` compute union, intersection, difference, and symmetric difference. Sets do not preserve order as a contract you should rely on for indexing; they are not sequences.

**Dicts.** Insertion ordered since 3.7. Keys must be hashable. `d[k]` raises `KeyError`; `d.get(k, default)` does not. Views from `keys()`, `values()`, and `items()` reflect later mutations.


In [8]:
nums = [3, 1, 2, 2]
nums.append(4)
print("sorted copy:", sorted(nums), "original:", nums)
nums.sort()
print("sorted in place:", nums)
print("slice:", nums[1:4], "reversed:", nums[::-1])

head, *middle, tail = [10, 20, 30, 40, 50]
print(head, middle, tail)

point = (4, 5)
x, y = point
print(x, y, (1,))

allowed = {"read", "write"}
print("read" in allowed)
print("union:", allowed | {"execute"})
print("difference:", allowed - {"write"})
print("intersection:", allowed & {"write", "admin"})

user = {"id": 7, "name": "Asha"}
user["role"] = "admin"
print(user.get("email", "missing"))
print(list(user.items()))

# dict as a frequency table
words = ["red", "blue", "red", "green", "blue", "red"]
freq = {}
for w in words:
    freq[w] = freq.get(w, 0) + 1
print(freq)


sorted copy: [1, 2, 2, 3, 4] original: [3, 1, 2, 2, 4]
sorted in place: [1, 2, 2, 3, 4]
slice: [2, 2, 3] reversed: [4, 3, 2, 2, 1]
10 [20, 30, 40] 50
4 5 (1,)
True
union: {'read', 'write', 'execute'}
difference: {'read'}
intersection: {'write'}
missing
[('id', 7), ('name', 'Asha'), ('role', 'admin')]
{'red': 3, 'blue': 2, 'green': 1}


## 9. Comprehensions, iterators, and generators

A comprehension builds a collection from an iterable with a filter and a transform, in one expression. List comprehensions produce a list. Set and dict comprehensions produce a set and a dict. Parentheses produce a generator expression, which yields values on demand.

The iteration protocol is two methods. `iter(obj)` calls `__iter__` and returns an iterator. `next(iterator)` calls `__next__` and returns the next item, or raises `StopIteration`. A `for` loop is that protocol plus automatic cleanup.

A generator function uses `yield`. Calling it does not run the body; it returns a generator object. Each `next` runs until the next `yield`, then suspends, preserving local state. Generators are the standard way to stream large or infinite sequences without holding them in memory.

`yield from` delegates to another iterable. Generator methods `send`, `throw`, and `close` exist for coroutines; for ordinary pipelines, plain `yield` is enough.


In [9]:
squares = [n * n for n in range(8) if n % 2 == 0]
print(squares)
unique_lengths = {len(w) for w in ["red", "blue", "red"]}
print(unique_lengths)
index = {w: i for i, w in enumerate(["red", "blue", "green"])}
print(index)

gen = (n * n for n in range(5))
print(next(gen), list(gen))

def countdown(start):
    while start > 0:
        yield start
        start -= 1

print(list(countdown(3)))

def flatten(rows):
    for row in rows:
        yield from row

print(list(flatten([[1, 2], [3], [4, 5]])))


[0, 4, 16, 36]
{3, 4}
{'red': 0, 'blue': 1, 'green': 2}
0 [1, 4, 9, 16]
[3, 2, 1]
[1, 2, 3, 4, 5]


## 10. Exceptions

An exception is an object raised to signal that the current block cannot continue. It is not a special return code. `try` marks the protected region. `except` handles a matching type. `else` runs if no exception escaped. `finally` runs on the way out, whether the block succeeded, failed, or returned.

Handle the narrowest type you can recover from. A bare `except:` also catches `KeyboardInterrupt` and `SystemExit`. Prefer `except SpecificError as exc`.

`raise` with no argument re-raises the current exception and preserves the traceback. `raise NewError(...) from exc` chains causes, which is the right pattern when translating a low-level failure into a domain failure.

Exceptions are for exceptional conditions, not for ordinary control flow that you expect on every call. "Key missing" inside a tight lookup may be cheaper as a membership test. "File missing" at the boundary is a fair use of `FileNotFoundError`.


In [10]:
def parse_age(raw):
    try:
        age = int(raw)
    except ValueError as exc:
        raise ValueError(f"age must be an integer, got {raw!r}") from exc
    else:
        if age < 0:
            raise ValueError("age cannot be negative")
        return age
    finally:
        pass

print(parse_age("30"))
try:
    parse_age("thirty")
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)
    print("cause:", type(exc.__cause__).__name__)


30
ValueError: age must be an integer, got 'thirty'
cause: ValueError


## 11. Files and context managers

A context manager defines `__enter__` and `__exit__`. The `with` statement calls `__enter__` before the block and `__exit__` afterwards, including when the block raises. File objects are context managers. The point is deterministic cleanup: the file is closed even if parsing fails halfway through.

Open text with an encoding. `open(path, encoding="utf-8")` is the portable form. Modes: `"r"` read text, `"w"` truncate and write, `"a"` append, `"x"` create exclusively, `"b"` binary. `"w"` destroys existing content.

`pathlib.Path` is the modern path API. It composes paths with `/`, reads and writes small files, and exposes metadata without string surgery.

For large files, iterate the file object. It yields lines and does not load the whole file. `read()` is acceptable for small configuration files.


In [11]:
from pathlib import Path

folder = Path("/tmp/python_mastery_demo")
folder.mkdir(exist_ok=True)
target = folder / "notes.txt"
target.write_text("alpha\nbeta\ngamma\n", encoding="utf-8")

with target.open(encoding="utf-8") as handle:
    lines = [line.rstrip("\n") for line in handle]
print(lines)
print(target.stat().st_size, "bytes")

class timed:
    def __enter__(self):
        import time
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc, tb):
        import time
        self.elapsed = time.perf_counter() - self.t0
        return False  # do not suppress exceptions

with timed() as clock:
    total = sum(range(1000))
print("sum:", total, "elapsed_set:", hasattr(clock, "elapsed"))


['alpha', 'beta', 'gamma']
17 bytes
sum: 499500 elapsed_set: True


## 12. Object-oriented programming

A class is a blueprint and a namespace. Instantiation calls `__new__` to allocate and `__init__` to initialise. Instance attributes live on the instance dictionary unless slots or properties say otherwise. Methods are functions whose first parameter is the instance, conventionally `self`.

`@classmethod` receives the class. Use it for alternative constructors. `@staticmethod` receives neither; it is a function parked in the class namespace for cohesion. `@property` exposes a method as an attribute and is the right place for a derived value or a guarded assignment.

Inheritance means a subclass can reuse and override behaviour. Method resolution is the MRO, visible as `Class.__mro__`. Call the parent implementation with `super()`, not by naming the parent, so multiple inheritance keeps working.

Special methods (`__repr__`, `__eq__`, `__len__`, `__iter__`, `__add__`, `__getitem__`) integrate your objects with the language. `__repr__` should be unambiguous. `__str__` may be friendly. If you define `__eq__`, consider `__hash__ = None` unless the object is immutable and the hash matches equality.

Dataclasses generate `__init__`, `__repr__`, and `__eq__` for record-like classes. They do not replace a real class when behaviour dominates data.


In [12]:
from dataclasses import dataclass

class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount
        return self._balance

    def __repr__(self):
        return f"Account(owner={self.owner!r}, balance={self._balance!r})"

class SavingsAccount(Account):
    def __init__(self, owner, balance=0, rate=0.02):
        super().__init__(owner, balance)
        self.rate = rate

    def apply_interest(self):
        self._balance += self._balance * self.rate
        return self._balance

acct = SavingsAccount("Asha", 1000)
acct.deposit(250)
print(acct, acct.apply_interest())

@dataclass(frozen=True)
class Point:
    x: float
    y: float

    def translated(self, dx, dy):
        return Point(self.x + dx, self.y + dy)

p = Point(1, 2)
print(p, p.translated(3, 4), hash(p))


Account(owner='Asha', balance=1275.0) 1275.0
Point(x=1, y=2) Point(x=4, y=6) -3550055125485641917


## 13. Decorators and functional tools

A decorator is a callable that accepts a function and returns a replacement. `@decorator` above `def` is syntactic sugar for `fn = decorator(fn)`. Use `functools.wraps` so the wrapper keeps the original name, docstring, and signature.

Decorators are the standard extension point for cross-cutting behaviour: logging, timing, caching, authorisation, and registration. A decorator factory is a function that returns a decorator, used when the decorator needs arguments.

`functools.lru_cache` memoises a pure function. `partial` freezes some arguments. `map` and `filter` exist; comprehensions are usually clearer. `sorted(items, key=...)` is the idiomatic way to order by a derived value.


In [13]:
import functools
import time

def trace(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"call {fn.__name__}{args}")
        return fn(*args, **kwargs)
    return wrapper

@trace
def area(width, height):
    """Return the area of a rectangle."""
    return width * height

print(area(3, 4), area.__name__, area.__doc__)

@functools.lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print([fib(n) for n in range(10)])

clamp = functools.partial(max, 0)
print([clamp(v) for v in [-2, 0, 5]])
print(sorted(["bb", "a", "ccc"], key=len))


call area(3, 4)
12 area Return the area of a rectangle.
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
[0, 0, 5]
['a', 'bb', 'ccc']


## 14. Modules and a curated standard library

A module is a file; a package is a directory with modules. `import module` executes the module once and binds the module object. Later imports reuse `sys.modules`. The idiom `if __name__ == "__main__"` lets a file be both an importable module and a script.

Reach for the standard library before adding a dependency.

- `collections.Counter`, `defaultdict`, `deque` for counting, defaulted maps, and double-ended queues
- `itertools` for lazy combinatorial pipelines
- `math` and `statistics` for numeric work that does not need NumPy
- `json` for text interchange
- `datetime` and `zoneinfo` for civil time
- `pathlib` for paths
- `re` for real pattern matching, not for every string split


In [14]:
from collections import Counter, defaultdict, deque
import itertools
import json
import math
from datetime import datetime, timezone

counts = Counter("abacab")
print(counts.most_common(2))

groups = defaultdict(list)
for word in ["ant", "bear", "ape"]:
    groups[word[0]].append(word)
print(dict(groups))

queue = deque([1, 2, 3])
queue.appendleft(0)
print(queue.popleft(), list(queue))

pairs = list(itertools.pairwise([10, 20, 30, 40]))
print(pairs)
print(round(math.hypot(3, 4), 2))

payload = json.dumps({"ok": True, "n": 3})
print(json.loads(payload))
print(datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"))


[('a', 3), ('b', 2)]
{'a': ['ant', 'ape'], 'b': ['bear']}
0 [1, 2, 3]
[(10, 20), (20, 30), (30, 40)]
5.0
{'ok': True, 'n': 3}
2026-10-05 16:30 UTC


## 15. Type hints and modern idioms

Annotations are optional metadata. CPython does not enforce them at runtime. They exist for readers, editors, and type checkers such as `mypy` or `pyright`. They pay off at module boundaries and in public functions.

Modern forms, from 3.9 onward, prefer built-in generics: `list[int]`, `dict[str, int]`. `X | None` replaces `Optional[X]` from 3.10. `tuple[int, ...]` means a variable-length tuple of ints.

Hints do not change behaviour. A function annotated as returning `int` can still return a string. Treat the checker as part of the build, not as a runtime guarantee.

Idioms worth keeping:
- Unpack, do not index, when the shape is fixed.
- Prefer comprehensions to `map` plus `lambda`.
- Use `enumerate` and `zip(..., strict=True)` instead of manual indexes.
- Use f-strings.
- Use `pathlib`.
- Use `logging`, not `print`, once the program has a real runtime.
- Keep functions small enough to name.


In [15]:
def mean(values: list[float]) -> float:
    if not values:
        raise ValueError("values must not be empty")
    return sum(values) / len(values)

def first_or(items: list[str], default: str | None = None) -> str | None:
    return items[0] if items else default

print(mean([2, 4, 6]))
print(first_or(["a", "b"]), first_or([]))
print(mean.__annotations__)

left = [1, 2, 3]
right = [10, 20, 30]
print([(a, b, a + b) for a, b in zip(left, right, strict=True)])


4.0
a None
{'values': list[float], 'return': <class 'float'>}
[(1, 10, 11), (2, 20, 22), (3, 30, 33)]


## 16. Capstone: a small, realistic object

The class below uses only the ideas above. It loads records, validates them, indexes them, and reports a summary. Nothing here requires a third-party library. The point is composition: a dataclass for the record, a dict for the index, a generator for lazy reading, exceptions for bad input, and a context manager for the file.


In [16]:
from dataclasses import dataclass
import json
from pathlib import Path

@dataclass(frozen=True)
class Reading:
    sensor: str
    value: float

    @classmethod
    def from_dict(cls, raw: dict) -> "Reading":
        try:
            return cls(sensor=str(raw["sensor"]), value=float(raw["value"]))
        except (KeyError, TypeError, ValueError) as exc:
            raise ValueError(f"bad reading: {raw!r}") from exc

class ReadingLog:
    def __init__(self):
        self._items: list[Reading] = []

    def add(self, reading: Reading) -> None:
        self._items.append(reading)

    def by_sensor(self, sensor: str) -> list[Reading]:
        return [r for r in self._items if r.sensor == sensor]

    def summary(self) -> dict[str, float]:
        totals: dict[str, list[float]] = {}
        for reading in self._items:
            totals.setdefault(reading.sensor, []).append(reading.value)
        return {name: sum(vals) / len(vals) for name, vals in totals.items()}

path = Path("/tmp/python_mastery_demo/readings.jsonl")
path.write_text(
    "\n".join([
        json.dumps({"sensor": "temp", "value": 21.5}),
        json.dumps({"sensor": "temp", "value": 22.0}),
        json.dumps({"sensor": "humidity", "value": 40}),
    ]) + "\n",
    encoding="utf-8",
)

log = ReadingLog()
with path.open(encoding="utf-8") as handle:
    for line in handle:
        log.add(Reading.from_dict(json.loads(line)))

print(log.summary())
print([r.value for r in log.by_sensor("temp")])


{'temp': 21.75, 'humidity': 40.0}
[21.5, 22.0]


## 17. A compact checklist

Before calling Python code finished, ask:

- Are names rebound, or is a shared mutable object being mutated?
- Are defaults immutable?
- Are exceptions specific, and is the cause chained when you translate them?
- Are files and other resources inside `with`?
- Does `__eq__` agree with `__hash__`, or is the object unhashable on purpose?
- Would a reader know the units, the empty case, and the failure case without running the code?
- Are type hints present on the public boundary?

Language fluency is the sum of those answers, not the number of syntax forms recalled.

---

*Notebook generated as a self-contained reference. Re-run the cells in order after a kernel restart.*
